In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_13_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 14 ###

# Preprocess 2018 and 2019 data without modifying originals
qi_no_q = 'On which platforms have you begun or completed data science courses'
qi_alt = 'On which online platforms have you begun or completed data science courses'

# Optimize 2018 modifications: single copy, in-place replace, vectorized column renaming
responses_2018_mod = responses_df_2018_cell10.copy()
responses_2018_mod.replace({
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}, inplace=True)
responses_2018_mod.columns = (
    responses_2018_mod.columns
    .str.replace(qi_alt, qi_no_q, regex=False)
    .str.replace('Kaggle Learn', 'Kaggle Learn Courses', regex=False)
    .str.replace('Fast.AI', 'Fast.ai', regex=False)
    .str.replace('Online University Courses', 'University Courses (resulting in a university degree)', regex=False)
)

# Optimize 2019 modifications similarly
responses_2019_mod = responses_df_2019_cell10.copy()
responses_2019_mod.replace({'Kaggle Courses (i.e. Kaggle Learn)': 'Kaggle Learn Courses'}, inplace=True)
responses_2019_mod.columns = (
    responses_2019_mod.columns
    .str.replace('Kaggle Courses (i.e. Kaggle Learn)', 'Kaggle Learn Courses', regex=False)
)

# Optimized functions
def grab_subset_of_data_26(original_df, question_of_interest):
    df = original_df.filter(like=question_of_interest, axis=1)
    mapping = {col: col.rsplit('-', 1)[-1].lstrip() for col in df.columns}
    return df.rename(columns=mapping).dropna(how='all')

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
        question_of_interest, include_2017=False):
    dfs_map = {
        '2022': responses_df_2022_cell10,
        '2021': responses_df_2021,
        '2020': responses_df_2020,
        '2019': responses_2019_mod,
        '2018': responses_2018_mod
    }
    years = ['2018', '2019', '2020', '2021', '2022']
    if include_2017:
        dfs_map['2017'] = responses_df_2017
        years.insert(0, '2017')
    # concatenate and add year index in C
    df_combined = pd.concat([
        grab_subset_of_data_26(dfs_map[y], question_of_interest) for y in years
    ], keys=years, names=['year']).reset_index(level='year').reset_index(drop=True)
    df_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_counts

def convert_df_of_counts_to_percentages_26(df, df_counts):
    totals = df.groupby('year').size()
    pct = df_counts.set_index('year').div(totals, axis=0) * 100
    return pct.reset_index()

# Combine, convert, and reshape
i_q = qi_no_q + '?'
learning_platform_df_combined, learning_platform_df_combined_counts = \
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(i_q)
learning_platform_df_combined_percentages = \
    convert_df_of_counts_to_percentages_26(
        learning_platform_df_combined, learning_platform_df_combined_counts
    )

# Clean and select columns
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns
    .str.replace('(resulting in a university degree)', '', regex=False)
)
cols = [
    'Coursera', 'University Courses ', 'Kaggle Learn Courses',
    'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai'
]
df_cell26 = (
    learning_platform_df_combined_percentages
    .loc[:, ['year'] + cols + ['None', 'Other']]
    .melt(id_vars='year', value_vars=cols)
    .sort_values(['year', 'value'], ascending=True)
)
# Rename variable column to blank
df_cell26.columns = df_cell26.columns.str.replace('variable', '', regex=False)
# Inspect final df
df_cell26.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 35 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    40 non-null     object 
 1           40 non-null     object 
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 155 ms, sys: 0 ns, total: 155 ms
Wall time: 155 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_14_try_1.pickle